In [5]:
import sys
import os
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../../source"))
from TDA_Testing import *
import pickle
def savepkl(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

def loadpkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

In [6]:
file_list = [
    "../../data/orbit_data/orbit5k_X_original_00.npy",
    "../../data/orbit_data/orbit5k_X_original_05.npy",
    "../../data/orbit_data/orbit5k_X_original_10.npy",
    "../../data/orbit_data/orbit5k_X_original_15.npy",
    "../../data/orbit_data/orbit5k_X_original_20.npy",
    "../../data/orbit_data/orbit5k_X_original_25.npy",
    "../../data/orbit_data/orbit5k_X_original_30.npy",
    "../../data/orbit_data/orbit5k_X_original_35.npy",
]
arrays = [np.load(f) for f in file_list]
X = np.stack(arrays, axis=0)
y = np.load("../../data/orbit_data/orbit5k_y.npy")

In [7]:
print(X.shape) 
print(y.shape)

(8, 5000, 1000, 2)
(5000,)


In [3]:
X_r1 = X[:,np.where(y==0)[0],:,:]
X_r2 = X[:,np.where(y==2)[0],:,:]
X_r3 = X[:,np.where(y==4)[0],:,:]

In [4]:
def compute_pd(X_subset):
    num_noise, num_samples = X_subset.shape[:2]
    pd_summary = [[[] for _ in range(num_samples)] for _ in range(num_noise)]
    
    for i in range(num_noise) :
        for j in range(num_samples):
            point_cloud = X_subset[i, j, :, :]  
            diagrams = ripser(point_cloud, maxdim=1)['dgms']
            h1_diag = diagrams[1]  # H1 diagram
            pd_summary[i][j] = h1_diag
    
    return pd_summary

In [12]:
# PD_r1=compute_pd(X_r1) 
# PD_r2=compute_pd(X_r2) 
# PD_r3=compute_pd(X_r3) 

#saving and loading 
savepkl(PD_r1, '../../data/orbit_data/PD_r1.pkl') 
savepkl(PD_r2, '../../data/orbit_data/PD_r2.pkl') 
savepkl(PD_r3, '../../data/orbit_data/PD_r3.pkl') 

# PD_r1 = loadpkl('../../data/orbit_data/PD_r1.pkl')
# PD_r2 = loadpkl('../../data/orbit_data/PD_r2.pkl')
# PD_r3 = loadpkl('../../data/orbit_data/PD_r3.pkl')

# Comparison p-values among test methods.

In [13]:
# 50 Data sampling 
rng = np.random.RandomState(1)
idx = rng.choice(1000, size=50, replace=False)
small_PD_r1 = [PD_r1[0][i] for i in idx]
small_PD_r2 = [PD_r2[0][i] for i in idx]
small_PD_r3 = [PD_r3[0][i] for i in idx]

rng = np.random.RandomState(2)
idx2 = rng.choice(1000, size=50, replace=False)
second_small_PD_r1 = [PD_r1[0][i] for i in idx2]
second_small_PD_r2 = [PD_r2[0][i] for i in idx2]
second_small_PD_r3 = [PD_r3[0][i] for i in idx2]

In [14]:
data_to_save = {
    "small_PD_r1": small_PD_r1, 
    "small_PD_r2": small_PD_r2,
    "small_PD_r3": small_PD_r3,
    "second_small_PD_r1": second_small_PD_r1, 
    "second_small_PD_r2": second_small_PD_r2,
    "second_small_PD_r3": second_small_PD_r3
}
# saving 
savepkl(data_to_save, '../../data/orbit_data/small_PD.pkl')

#### Aggtest

In [7]:
#linear weight
func_weight = function_weight("Poly", poly_order=1)
Agg_result_r1r2=Aggtest(small_PD_r1,small_PD_r2,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r1r3=Aggtest(small_PD_r1,small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r2r3=Aggtest(small_PD_r2,small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)

Agg_result_r1r1=Aggtest(small_PD_r1,second_small_PD_r1,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r2r2=Aggtest(small_PD_r2,second_small_PD_r2,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r3r3=Aggtest(small_PD_r3,second_small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)


In [8]:
minimum_pval_linear=[]
for result in [Agg_result_r1r1,Agg_result_r1r2,Agg_result_r1r3,Agg_result_r2r2,Agg_result_r2r3,Agg_result_r3r3]:
        minimum_pval_linear.append(result[1]['p-value'])
minimum_pval_linear

[0.8981018981018981,
 0.000999000999000999,
 0.001998001998001998,
 0.22677322677322678,
 0.000999000999000999,
 0.3086913086913087]

In [9]:
#constant weight
func_weight = function_weight("constant")
Agg_result_r1r2=Aggtest(small_PD_r1,small_PD_r2,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r1r3=Aggtest(small_PD_r1,small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r2r3=Aggtest(small_PD_r2,small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)

Agg_result_r1r1=Aggtest(small_PD_r1,second_small_PD_r1,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r2r2=Aggtest(small_PD_r2,second_small_PD_r2,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r3r3=Aggtest(small_PD_r3,second_small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)


In [10]:
minimum_pval_constant=[]
for result in [Agg_result_r1r1,Agg_result_r1r2,Agg_result_r1r3,Agg_result_r2r2,Agg_result_r2r3,Agg_result_r3r3]:
    minimum_pval_constant.append(result[1]['p-value'])
minimum_pval_constant

[0.984015984015984,
 0.001998001998001998,
 0.003996003996003996,
 0.14685314685314685,
 0.001998001998001998,
 0.5244755244755245]

In [11]:
#arctan weight
func_weight = function_weight("arctan")
Agg_result_r1r2=Aggtest(small_PD_r1,small_PD_r2,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r1r3=Aggtest(small_PD_r1,small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r2r3=Aggtest(small_PD_r2,small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)

Agg_result_r1r1=Aggtest(small_PD_r1,second_small_PD_r1,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r2r2=Aggtest(small_PD_r2,second_small_PD_r2,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)
Agg_result_r3r3=Aggtest(small_PD_r3,second_small_PD_r3,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True,return_dictionary=True)


In [12]:
minimum_pval_arctan=[]
for result in [Agg_result_r1r1,Agg_result_r1r2,Agg_result_r1r3,Agg_result_r2r2,Agg_result_r2r3,Agg_result_r3r3]:
    minimum_pval_arctan.append(result[1]['p-value'])
minimum_pval_arctan

[0.6153846153846154,
 0.000999000999000999,
 0.000999000999000999,
 0.19480519480519481,
 0.000999000999000999,
 0.7222777222777222]

#### PD  test

In [13]:
_,_, PDp_val_r1r2 = permutation_test(small_PD_r1 , small_PD_r2, num_permutations=10**3)
_,_, PDp_val_r1r3 = permutation_test(small_PD_r1 , small_PD_r3, num_permutations=10**3)
_,_, PDp_val_r2r3 = permutation_test(small_PD_r2 , small_PD_r3, num_permutations=10**3)
_,_, PDp_val_r1r1 = permutation_test(small_PD_r1 , second_small_PD_r1, num_permutations=10**3)
_,_, PDp_val_r2r2 = permutation_test(small_PD_r2 , second_small_PD_r2, num_permutations=10**3)
_,_, PDp_val_r3r3 = permutation_test(small_PD_r3 , second_small_PD_r3, num_permutations=10**3)

PDp_val = [PDp_val_r1r1,PDp_val_r1r2,PDp_val_r1r3,PDp_val_r2r2,PDp_val_r2r3,PDp_val_r3r3]

In [14]:
PDp_val

[0.7832167832167832,
 0.000999000999000999,
 0.000999000999000999,
 0.17182817182817184,
 0.000999000999000999,
 0.5874125874125874]

In [19]:
savepkl(PDp_val, '../../results/simulation_results/Orbit5k_results/orbit5k_PD_result.pkl')

#### PL test

In [15]:
#generating persistence landscape
small_PL_r1 = [ PersLandscapeApprox(dgms=[dgm]) for dgm in small_PD_r1]
small_PL_r2 = [ PersLandscapeApprox(dgms=[dgm]) for dgm in small_PD_r2]
small_PL_r3 = [ PersLandscapeApprox(dgms=[dgm]) for dgm in small_PD_r3]
second_small_PL_r1 = [ PersLandscapeApprox(dgms=[dgm]) for dgm in second_small_PD_r1]
second_small_PL_r2 = [ PersLandscapeApprox(dgms=[dgm]) for dgm in second_small_PD_r2]
second_small_PL_r3 = [ PersLandscapeApprox(dgms=[dgm]) for dgm in second_small_PD_r3]

In [16]:
PLp_val_r1r2 = permutation_pl_test(small_PL_r1 , small_PL_r2)
PLp_val_r1r3 = permutation_pl_test(small_PL_r1 , small_PL_r3)
PLp_val_r2r3 = permutation_pl_test(small_PL_r2 , small_PL_r3)
PLp_val_r1r1 = permutation_pl_test(small_PL_r1 , second_small_PL_r1)
PLp_val_r2r2 = permutation_pl_test(small_PL_r2 , second_small_PL_r2)
PLp_val_r3r3 = permutation_pl_test(small_PL_r3 , second_small_PL_r3)

In [17]:
PLp_val = [PLp_val_r1r1,PLp_val_r1r2,PLp_val_r1r3,PLp_val_r2r2,PLp_val_r2r3,PLp_val_r3r3]

In [18]:
PLp_val 

[0.6713286713286714,
 0.000999000999000999,
 0.000999000999000999,
 0.8441558441558441,
 0.000999000999000999,
 0.7292707292707292]

In [20]:
savepkl(PLp_val, '../../results/simulation_results/Orbit5k_results/orbit5k_PL_result.pkl')

In [21]:
pi_result=pd.read_csv("../../results/simulation_results/Orbit5k_results/orbit5k_PI_result.csv")
PIp_val_constant = pi_result.loc[0][1:7].values
PIp_val_linear = pi_result.loc[0][7:13].values
PIp_val_arctan = pi_result.loc[0][13:20].values

In [22]:
methods = {
    "Agg_Linear": minimum_pval_linear,
    "Agg_Arctan": minimum_pval_arctan,
    "Agg_Constant": minimum_pval_constant,
    "PI_Linear": PIp_val_linear,
    "PI_Arctan": PIp_val_arctan,
    "PI_Constant": PIp_val_constant ,
    "PD": PDp_val ,
    "PL": PLp_val 
}

df = pd.DataFrame.from_dict(methods, orient='index', columns=[
    "Scenario1", "Scenario2", "Scenario3", "Scenario4", "Scenario5", "Scenario6"
])


df = df[['Scenario1','Scenario4','Scenario6','Scenario2','Scenario3','Scenario5']]
df = df.rename(columns={'Scenario4':'Scenario2',
                        'Scenario6':'Scenario3',
                        'Scenario2':'Scenario4',
                        'Scenario3':'Scenario5',
                        'Scenario5':'Scenario6'})

df.reset_index(inplace=True)
df.rename(columns={'index':'Method'}, inplace=True)

df.to_csv("../../results/simulation_results/Orbit5k_results/orbit5k_pvalues.csv", index=False)


In [23]:
df

,Method,Scenario1,Scenario2,Scenario3,Scenario4,Scenario5,Scenario6
0,Agg_Linear,0.898102,0.226773,0.308691,9.990010e-04,1.998002e-03,9.990010e-04
1,Agg_Arctan,0.615385,0.194805,0.722278,9.990010e-04,9.990010e-04,9.990010e-04
2,Agg_Constant,0.984016,0.146853,0.524476,1.998002e-03,3.996004e-03,1.998002e-03
3,PI_Linear,0.844892,0.996853,0.380434,1.606329e-10,3.108441e-25,1.537986e-16
4,PI_Arctan,0.999564,0.999892,0.398920,1.072634e-04,1.869999e-03,4.947125e-07
5,PI_Constant,0.999864,0.570245,0.504767,1.671198e-01,2.615973e-02,1.751058e-04
6,PD,0.783217,0.171828,0.587413,9.990010e-04,9.990010e-04,9.990010e-04
7,PL,0.671329,0.844156,0.729271,9.990010e-04,9.990010e-04,9.990010e-04


In [24]:
#df=pd.read_csv("../../results/simulation_results/Orbit5k_results/orbit5k_pvalues.csv")

In [26]:
def df_to_custom_latex(df, caption="p-values table", label="tab:sim"):
    """
    Convert your pandas DataFrame to the desired LaTeX format.
    Assumes the first column is 'Method' and the rest are Scenario columns.
    """

    # Define method groups
    groups = {
        "Agg_": "Agg Intensity",
        "PI_": "PI",
        "PD": "PD",
        "PL": "PL"
    }

    # LaTeX output construction
    latex = []
    latex.append("\\begin{table}[H]")
    latex.append("\\centering")
    latex.append(f"\\caption{{{caption}}}")
    latex.append(f"\\label{{{label}}}")

    # Header
    scenarios = df.columns[1:]
    header = " & ".join(scenarios)
    latex.append(f"\\begin{{tabular}}{{l{'c' * len(scenarios)}}}")
    latex.append("\\toprule")
    latex.append(" & " + " & ".join(scenarios) + " \\\\")
    latex.append("\\midrule")

    # Populate LaTeX table rows
    for key, group_name in groups.items():

        # Add group label (skip for PD and PL since they are single rows)
        if key in ["PD", "PL"]:
            pass
        else:
            latex.append(f"\\multicolumn{{{len(scenarios)}}}{{l}}{{\\textbf{{{group_name}}}}} \\\\")

        # Iterate through rows of DataFrame
        for _, row in df.iterrows():
            method = row["Method"]

            # Handle PD (single row)
            if key == "PD" and method == "PD":
                vals = []
                for v in row[1:]:
                    vals.append("<0.001" if v < 0.01 else f"{v:.3f}")
                latex.append(f"\\textbf{{PD}} & " + " & ".join(vals) + " \\\\")
                continue

            # Handle PL (single row)
            if key == "PL" and method == "PL":
                vals = []
                for v in row[1:]:
                    vals.append("<0.001" if v < 0.01 else f"{v:.3f}")
                latex.append(f"\\textbf{{PL}} & " + " & ".join(vals) + " \\\\")
                continue

            # Handle Agg_ and PI_ prefixes
            if method.startswith(key):
                # Convert method name (e.g., Agg_Linear → Linear)
                sub = method.replace(key, "")
                vals = []
                for v in row[1:]:
                    vals.append("<0.001" if v < 0.01 else f"{v:.3f}")
                latex.append(f"\\quad {sub} & " + " & ".join(vals) + " \\\\")

        latex.append("")  # Blank line between groups

    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")

    return "\n".join(latex)



In [27]:
latex_code = df_to_custom_latex(
    df,
    caption="p-values of four testing methods of orbit5k datasets.",
    label="tab:simulation2"
)

print(latex_code)

\begin{table}[H]
\centering
\caption{p-values of four testing methods of orbit5k datasets.}
\label{tab:simulation2}
\begin{tabular}{lcccccc}
\toprule
 & Scenario1 & Scenario2 & Scenario3 & Scenario4 & Scenario5 & Scenario6 \\
\midrule
\multicolumn{6}{l}{\textbf{Agg Intensity}} \\
\quad Linear & 0.898 & 0.227 & 0.309 & <0.001 & <0.001 & <0.001 \\
\quad Arctan & 0.615 & 0.195 & 0.722 & <0.001 & <0.001 & <0.001 \\
\quad Constant & 0.984 & 0.147 & 0.524 & <0.001 & <0.001 & <0.001 \\

\multicolumn{6}{l}{\textbf{PI}} \\
\quad Linear & 0.845 & 0.997 & 0.380 & <0.001 & <0.001 & <0.001 \\
\quad Arctan & 1.000 & 1.000 & 0.399 & <0.001 & <0.001 & <0.001 \\
\quad Constant & 1.000 & 0.570 & 0.505 & 0.167 & 0.026 & <0.001 \\

\textbf{PD} & 0.783 & 0.172 & 0.587 & <0.001 & <0.001 & <0.001 \\

\textbf{PL} & 0.671 & 0.844 & 0.729 & <0.001 & <0.001 & <0.001 \\

\bottomrule
\end{tabular}
\end{table}
